In [1]:
import pandas as pd
import numpy as np

In [2]:
games = pd.read_csv("games.csv")

games.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


In [3]:
games.columns

Index(['appid', 'name', 'release_date', 'english', 'developer', 'publisher',
       'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags',
       'achievements', 'positive_ratings', 'negative_ratings',
       'average_playtime', 'median_playtime', 'owners', 'price'],
      dtype='object')

In [5]:
#select usefull columns
games = games[['name',
               'genres',
               'categories',
               'steamspy_tags',
               'positive_ratings']]

In [6]:
#remove missing values 
games.isnull().sum()

name                0
genres              0
categories          0
steamspy_tags       0
positive_ratings    0
dtype: int64

In [7]:
games.dropna(inplace=True)

In [8]:
#Create tag columns 
games['tags'] = (
    games['genres'] +
    " " +
    games['categories'] +
    " " +
    games['steamspy_tags']
)

In [9]:
new_games = games[['name','tags']]

In [10]:
#convert text into numbers 
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_games['tags']).toarray()
vectors.shape

(27075, 404)

In [11]:
#calculate similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [12]:
#recommendation function 
def recommend(game):

    game_index = new_games[new_games['name'] == game].index[0]

    distances = similarity[game_index]

    games_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in games_list:
        print(new_games.iloc[i[0]].name)

In [13]:
recommend("Dota 2")

6810
20472
7926
13875
14926


In [14]:
import pickle
pickle.dump(
    new_games,
    open('games.pkl','wb')
)
pickle.dump(
    similarity,
    open('similarity.pkl','wb')
)